# Preparation for creating simulated data
## Aim of this notebook
This notebook contains the preparation required to run amplicon-sequencing (AmpSeq) read simulations using the read simulation pipeline "pop_var_sim", provided in a [separate repository](https://github.com/ISOinaBox/pop_var_sim).  

While the files produced in this notebook can be used to create simulated reads, the main use of this notebook is to serve as a template and guide for generating simulation runs, tailor-made for a pipeline that is to be validated.  

This notebook focuses on AmpSeq data because simulated data is particularly useful for this type of data. Whole-genome data for pipeline validataion can be obtained from real-world datasets that are readily available [from public resources](https://www.malariagen.net/resource/36/). AmpSeq data, on the other hand, depends on a specific primer panel, for which data may not be readily available. In addition, a pipeline written and validated for a specific AmpSeq panel, such as [SpotMalaria](https://ngs.sanger.ac.uk/production/malaria/Resource/29/20200705-GenRe-04a-SpotMalaria-0.39.pdf) may need to be validated against other primer panels, for which data does not exist yet.  

In Part A of this notebook, we use the Pf8 drug resistance rules to "design" a set of sample genotypes that provide broad coverage of the various drug resistance classification scenarios, that we would like to use pop_sim_var to simulate.  

In part B, we prepare the files for pop_var_sim to be run. We first obtain read counts from high-quality public data. This step is necessary for any simulation, so that the generated data provides realistic input for an analysis pipeline. Here, we simulate a [SpotMalaria](https://ngs.sanger.ac.uk/production/malaria/Resource/29/20200705-GenRe-04a-SpotMalaria-0.39.pdf) AmpSeq run, for which real-world samples exist in public repositories. This may not always be the case for simulating different AmpSeq primer panel runs, in which case the user may need to estimate realistic read counts in different ways. Secondly, the simulation run input files are created. This demonstrates how genome data and publicly available data on relevant mutations can be used to simulate WT and mutant strain results and can be modified according to the validation that is to be carried out. For example, a new pipeline might need to be tested in terms of how robustly it can call a mutant allele in a mix of WT and mutant parasites, which can be configured in the simulation pipeline run.  

## Part A - Defining a represent set of synthetic samples for Malaria drug resistance. 

Our main information source for this will be the Pf8 [documentation on resistance classification](https://pf8-release.cog.sanger.ac.uk/Pf8_resistance_classification.pdf), supplemented by other data where required/appropriate

### Sample design per drug

#### Artemisinin

From the Pf8 report, resistance is determined by non-synonymous mutations in the BTB/POZ and propellor domains of kelch13. However, in order to create realistic scenarios for synthetic samples, we will in addition use the [WHO technical report on artemisinin](https://iris.who.int/server/api/core/bitstreams/be1df79e-560f-4200-a0b3-d404f7d46330/content), which identifies specific mutations as validated for resistance association, and a further 2 that in isolation do not confer resistance. 

Excluding / ignoring the “Missing” genotypes, we therefore propose the following set of samples that cover the range of statuses:


|Genotype PF3D7_1343700 (kelch13)|Source|Resistance status|Notes|
|---|---|---|---|
|WT||Sensitive||
|N458Y|WHO|Resistant||
|Y493H|WHO|Resistant||
|R539T|WHO|Resistant||
|I543T|WHO|Resistant||
|R561H|WHO|Resistant||
|C580Y|WHO|Resistant||
|E252Q|WHO|Sensitive||
|A578S|WHO|Sensitive||
|E252Q,C580Y,A578S|WHO+Pf8|Resistant||
|N458Y,Y493H,R539T,I543T,R561H,C580Y / WT,WT,WT,WT,WT,WT|WHO+Pf8|Undetermined|This sample is designed as a test; all of the 6 validated mutants are present, but they are all heterozygous; the sample should therefore be classified as Undetermined|

#### Chloroquine
Baed on Pf8 rules, we will restrict attention to the crt K76 allele, and propose the following samples to cover the various classification scenarios:

|Genotype PF3D7_0709000 (crt)|Resistance status|
|---|---|
|WT|Sensitive|
|K76T/WT|Undermined|
|K76T|Resistant
|K76[any non-T aa]|Undetermined|

#### Pyrimethamine

Using the Pf8 rules, we will restrict attention to the dhfr S108 allele, and propose the following set of synthetic samples that cover the classification scenarios (once again ignoring scenarios that refer to “Missing” genotypes):

|Genotype PF3D7_0417200 (dhfr)|Resistance status|
|---|---|
|WT|Sensitive|
|S108N/WT|Undetermined|
|S108N|Resistant|
|S108[any non-n aa]|Undetermined|

#### Sufadoxine
Using the Pf8 rules, we will restrict attention to the dhps A437 allele, and propose the following set of synthetic samples that cover the classification scenarios (once again ignoring scenarios that refer to “Missing” genotypes):

|Genotype PF3D7_0810800 (dhps)|Resistance status|
|---|---|
|WT|Sensitive|
|A437G/WT (het)|Undetermined|
|A437G|Resistant|
|A437[any non-G aa]|Undetermined|


#### S-P, S-P-IPTp

The Pf8 rules are defined by specific combinations of 3 alleles of the dhfr gene (associated with resistance in standard treatment) and further alleles of the dhps gene (which, when combined with the aforementioned dhfr mutations, are associated with super-resistance in the now more common intermittent treatment during pregnancy); we propose these samples that cover key scenarios. 

|Genotype PF3D7_0417200 (dhfr), PF3D7_0810800 (dhps)|Resistance status|Notes|
|---|---|---|
|dhfr: N51I, C59R, S108N|Resistant|Standard resistant Triple-mutant|
|dhrf: N51I, C59R, S108N, I164L; dhps: A437G, K540E, A581G, A613S|Resistant|All listed mutations, super-resistance|
|dhrf: N51I, C59R, S108N; dhps: A437G, K540E, A613T|Resistant|Minimal 6 required mutations for super-resistance|
|dhrf: C59R, S108N, I164L; dhps: A437G, K540E, A613T|Sensitive|6 mutations, but lacking one of the compulsory set of 5|


#### Mefloquine, Piperaquine, AS-MQ, DHA-PPQ

These require genotyping of copy number of mdr1 and plasmepsin (respectively), which is beyond the current capabilities of the pop_var_sim pipeline, so will not be pursued further. 

## Putting it together

Taking the above descriptions, we formulate the following list of synthetic samples, using a file format similar to that produced for the real-world data set

In [1]:
import pandas as pd

df = pd.read_csv('malaria_DR_synthetic_samples.csv')
df

,sample,dhfr_51,dhfr_59,dhfr_108,dhfr_164,crt_76,dhps_437,dhps_540,dhps_581,dhps_613,...,kelch13_543,kelch13_561,kelch13_580,kelch13_252,kelch13_578,Artemisinin,Chloroquine,Pyrimethamine,Suladoxine,S-P
0,WT,N,C,S,I,K,A,K,A,A,...,I,R,C,E,A,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
1,Ar_k13_N458Y,N,C,S,I,K,A,K,A,A,...,I,R,C,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
2,Ar_k13_Y493H,N,C,S,I,K,A,K,A,A,...,I,R,C,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
3,Ar_k13_R539T,N,C,S,I,K,A,K,A,A,...,I,R,C,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
4,Ar_k13_I543T,N,C,S,I,K,A,K,A,A,...,T,R,C,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
5,Ar_k13_R561H,N,C,S,I,K,A,K,A,A,...,I,H,C,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
6,Ar_k13_C580Y,N,C,S,I,K,A,K,A,A,...,I,R,Y,E,A,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
7,Ar_k13_E252Q,N,C,S,I,K,A,K,A,A,...,I,R,C,Q,A,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
8,Ar_k13_A578S,N,C,S,I,K,A,K,A,A,...,I,R,C,E,S,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
9,Ar_k13_multi1,N,C,S,I,K,A,K,A,A,...,I,R,Y,Q,S,Resistant,Sensitive,Sensitive,Sensitive,Sensitive


## Part B: Configuring the read simulation pipeline
The [read simulation pipeline](https://github.com/ISOinaBox/pop_var_sim) requires a set of input files to configure the simulation run. They are:

* A reference genome and a collection of VCF-like files, each encoding a "genotype", or variant form of the reference genome we wish to simulate samples from
* A "haplotype manifest file" that contains a list of virtual PCR haplotypes, each combining a reference genome, a VCF and an list of amplicon postions on the reference genome
* A "sample design" file, that defines the samples we wish to generate synthetically, by combining haploypes in the manifest file in appropriate proportions. We also need to define the number of reads (pairs) to generate for each synthetic sample.  

We will now gather the various pieces we need.

### 1. Read counts to guide simulation
A consideration for a simulated dataset is the number of reads to create. This should be within the range of expected read numbers from real-world experiments because unrealistically high or low read counts will not provide useful insights into the validity of pipeline results.  

Our task is to simulate reads from PCR amplification of genomic DNA in _P. falciparum_ samples using the [SpotMalaria](https://www.malariagen.net/project/spotmalaria/) panel, as used for the [GenRe Mekong project](https://www.malariagen.net/resource/29/) project. This primer panel consists of three sub-panels: SPEC (speciation), GRC1 and GRC2.  The data for these panels is provided in [this spreadsheet](https://www.malariagen.net/wp-content/uploads/2023/11/20200705-GenRe-04b-SpotMalaria-SupplementaryFile1.xls).  

As noted in the introduction, this analysis is specific to the SpotMalaria AmpSeq primer panel and will need to be modified - based on the template provided here - to accommodate other AmpSeq panels or whole-genome data.  

For the purpose of this analysis, the dataset [Pf8-GenReMekong_concordant_phenotype_high_quality_samples.csv](../real-world_gold-standard_data/Pf8-GenReMekong/Pf8-GenReMekong_concordant_phenotype_high_quality_samples.csv), created [here](../real-world_gold-standard_data/Pf8-GenReMekong/pf8genre.ipynb#Save-results:-concordant-phenotype-data) was used. This contains a subset of 904 samples of all high-quality data from the GenRe Mekong project where drug resistance phenotype agrees with the WGS sample analysis in Pf8.  

Using the dataset [Pf8-GenReMekong_concordant_phenotypes_allcols.csv](../../../real-world_gold-standard_data/Pf8-GenReMekong/additional_output_files/Pf8-GenReMekong_concordant_phenotypes_allcols.csv) create read count stats for the three SpotMalaria amplicon panels.  

In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [3]:
samples_df = pd.read_csv('../../real-world_gold-standard_data/Pf8-GenReMekong/additional_output_files/Pf8-GenReMekong_concordant_phenotypes_allcols.csv')
samples_df

,sample,country,ampseq_process,Artemisinin,Piperaquine,Mefloquine,Chloroquine,Pyrimethamine,Sulfadoxine,DHA-PPQ,...,INSDC_GenRe_SPEC,INSDC_GenRe_SPEC_readcount,GenRe_GRC1_ENA_FASTQ_FTP_1,GenRe_GRC2_ENA_FASTQ_FTP_1,GenRe_SPEC_ENA_FASTQ_FTP_1,Pf8_ENA_FASTQ_FTP_1,GenRe_GRC1_ENA_FASTQ_FTP_2,GenRe_GRC2_ENA_FASTQ_FTP_2,GenRe_SPEC_ENA_FASTQ_FTP_2,Pf8_ENA_FASTQ_FTP_2
0,RCN12025,Vietnam,AmpSeqV1,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,...,ERR14388605,1094.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/003/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/004/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/005/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/006/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/003/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/004/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/005/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/006/...
1,RCN12026,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,...,ERR14388608,1192.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/006/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/007/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/008/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/007/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/006/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/007/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/008/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/007/...
2,RCN12028,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,...,ERR14388614,1043.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/012/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/013/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/014/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/009/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/012/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/013/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/014/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/009/...
3,RCN12031,Vietnam,AmpSeqV1,Resistant,Resistant,Sensitive,Resistant,Resistant,Resistant,Resistant,...,ERR14388623,293.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/021/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/022/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/023/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/010/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/021/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/022/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/023/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/010/...
4,RCN12032,Vietnam,AmpSeqV1,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,...,ERR14388626,960.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/024/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/025/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/026/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/011/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/024/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/025/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/026/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/011/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899,RCN26767,Laos,AmpSeqV2,Resistant,Sensitive,Sensitive,Resistant,Resistant,Resistant,Sensitive,...,ERR14397548,766.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/046/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/047/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/048/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/052/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/046/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/047/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/048/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/052/...
900,RCN26775,Laos,AmpSeqV2,Sensitive,Sensitive,Sensitive,Resistant,Resistant,Sensitive,Sensitive,...,ERR14397572,734.0,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/070/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/071/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/072/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR156/053/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/070/...,ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR143/071/...,ftp://ftp

Handily, the read counts have already been collected from ENA in the above table. So it is now a case of aggergating over them. 

In [4]:
readcount_cols = ['INSDC_GenRe_GRC1_readcount', 'INSDC_GenRe_GRC2_readcount', 'INSDC_GenRe_SPEC_readcount',]
stats = (
    samples_df[readcount_cols]
        .describe(percentiles=[0.25, 0.5, 0.75])
        .loc[["min", "25%", "50%", "75%", "max"]]
)

stats

,INSDC_GenRe_GRC1_readcount,INSDC_GenRe_GRC2_readcount,INSDC_GenRe_SPEC_readcount
min,18.00,21.00,3.00
25%,25556.75,26709.75,655.25
50%,32392.50,32461.50,987.50
75%,40280.75,39863.25,1443.25
max,96442.00,78617.00,17924.00


The read count stats shown above can be used as guidelines for simulation runs. Looking at the median, counts of 32,000 for GRC1 and GRC2, and 1000 for SPEC, seem sensible. 

### 2. Create a bed-like file of primer positions
A file of primer positions in a bed-like format is rquired to configure start/end positions of amplicons to simulate.  
As we are simulating a SpotMalaria AmpSeq run, we are using primer positions for the GenRe-04b-SpotMalaria AmpSeq panel, available from www.malariagen.net in form of an Excel file. 

In [5]:
panel_df = pd.read_excel(
    "https://www.malariagen.net/wp-content/uploads/2023/11/20200705-GenRe-04b-SpotMalaria-SupplementaryFile1.xlsx",
    sheet_name="P. falciparum amplicon primers"
)

Change column names in line with expected headers for bed-like config file for the simulation pipeline.  Add a column DESC, expected by the simulation pipeline. Use it to record the primer panel ('GRC1','GRC2' or 'SPEC') and drop rows (if any) that do not have a primer panel assigned. Only keep the columns relevant to the simulation pipeline.

In [6]:
panel_df.rename(
    columns={
        'Chromosome':'#CHROM', 
        'Amplicon Start Position':'START',
        'Amplicon Stop Position':'END'
    },inplace=True)

panel_df['DESC'] = panel_df.apply(
    lambda row:
    'SPEC' if 'SPEC' in row['Multiplex'] else 
    'GRC1' if 'GRC1' in row['Multiplex'] else 
    'GRC2' if 'GRC2' in row['Multiplex'] else
    np.nan, 
    axis=1
)
panel_df=panel_df[panel_df['DESC'].notna()]
keep_cols=['#CHROM','START','END','DESC']
panel_df=panel_df[keep_cols]

Coordinates are 1-based in this file but need to be converted to half-open BED coords for the simulation pipeline. We also need to make a couple of corrections to the data:
* Incorrect start coord is of 298737 is given for the amplicon "Plas23_breakpoint" (which is greater than the end coord). Correct value for this should be 289567;  
* The mitochondrial sequences is referred to as "Pf_M76611" in the panel files, but as "Pf3D7_MIT_v3" in the GeneDB/Ensembl reference genome files. Change its name here to save problems down the line

In [7]:
panel_df['#CHROM']=panel_df['#CHROM'].replace("Pf_M76611", "Pf3D7_MIT_v3")
panel_df.loc[(panel_df["#CHROM"] == "Pf3D7_14_v3") & (panel_df["START"] == 298737), "START"] = 289567
panel_df['START']=pd.to_numeric(panel_df['START'],errors='coerce')-1

Keep only rows where primer start and end is provided

In [8]:
panel_df = panel_df[(panel_df['START'].notna()) & (panel_df['END'].notna())]

convert positions to integer

In [9]:
panel_df['START']=panel_df['START'].astype(np.int64)
panel_df['END']=panel_df['END'].astype(np.int64)
panel_df

,#CHROM,START,END,DESC
0,Pf3D7_MIT_v3,88,315,SPEC
1,Pf3D7_MIT_v3,1637,1893,SPEC
2,Pf3D7_01_v3,145386,145591,GRC1
3,Pf3D7_01_v3,179274,179472,GRC2
4,Pf3D7_01_v3,180439,180684,GRC1
...,...,...,...,...
132,Pf3D7_14_v3,2480904,2481117,GRC1
133,Pf3D7_14_v3,2625793,2626012,GRC2
134,Pf3D7_14_v3,2733564,2733804,GRC1
135,Pf3D7_14_v3,3045934,3046142,GRC2


Create versions of the panel file for each of the three sub-panels and save files.

In [10]:
spec_panel_df = panel_df[panel_df['DESC']=='SPEC']
grc1_panel_df = panel_df[panel_df['DESC']=='GRC1']
grc2_panel_df = panel_df[panel_df['DESC']=='GRC2']

spec_panel_df.to_csv('../simulation_run_input_files/SpotMalaria-SPEC_Pf_amplicon_primers.bed', index=False, sep="\t")
grc1_panel_df.to_csv('../simulation_run_input_files/SpotMalaria-GRC1_Pf_amplicon_primers.bed', index=False, sep="\t")
grc2_panel_df.to_csv('../simulation_run_input_files/SpotMalaria-GRC2_Pf_amplicon_primers.bed', index=False, sep="\t")

### 3. Reference genome and annotation
The read simulation pipeline requires a reference genome. The _P. falciparum_ 3D7 v3 reference genome was obtained from PlasmoDB [here](https://plasmodb.org/common/downloads/Current_Release/Pfalciparum3D7/fasta/data/PlasmoDB-68_Pfalciparum3D7_Genome.fasta)
and is provided in folder ```../../simulation_run_input_files/```.  

In addition, a GFF genome annotation file is required for the next step, creating a variant file. It was obtained from 
https://plasmodb.org/a/service/raw-files/release-68/Pfalciparum3D7/gff/data/PlasmoDB-68_Pfalciparum3D7.gff and is also provided in the same folder.


### 4. Remaining files: haplotype VCFs, haplotype manifest and sample design files

In part A of this notebook, we created a high-level sample design file (CSV). We will now use this file to configure pop_var_sim to simulate synthetic collections of reads for the samples in the file, taking into account the fact that we will create 3 synthetic sequencing products for each sample (one each for the 3 sub-panels the SPOT Malaria).

The majority of work required for this task is to convert the protein alleles defined in the file into genomic variants required by pop_var_sim. We have written a script to automate this process and produce the files that pop_var_sim needs. The script does the following:

* Identify the genomic codon for each protein allele in the file;
* If the given AA allele does not match the genome, determine the SNP that would give rise to the defined AA allele;
* For each listed sample, and using the function above, produce a VCF that encodes the genome that would give rise to that sample;
* Produce the required manifest combining the input panel files and VCFs into synthetic PCR haplotypes; 
* Produce the sample design file corresponding to the input CSV, by putting together the PCR haplotypes in the appropriate combinations. 


Note that the script writes its output files to the current working directory. So we do not run it from this notebook, but instead show how it was run to generate the files included in this repo:

cd ../simulation_run_input_files

python ../prepare_simulation_run/prepare_pop_var_sim_files.py --fasta PlasmoDB-68_Pfalciparum3D7_Genome.fasta --gff PlasmoDB-68_Pfalciparum3D7.gff --csv ../prepare_simulation_run/malaria_DR_synthetic_samples.csv --bed_grc1 SpotMalaria-GRC1_Pf_amplicon_primers.bed --bed_grc2 SpotMalaria-GRC2_Pf_amplicon_primers.bed --bed_spec SpotMalaria-SPEC_Pf_amplicon_primers.bed --reads_grc1 32000 --reads_grc2 32000 --reads_spec 1000 

 